In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_2")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]

condition_pal = {
    "wt": blues[2],
    "bcd": reds[2],
    "trk": greens[2],
}

In [ ]:
ap_vals = np.linspace(0.001, 0.98, 6)

In [ ]:
all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

all_surface_areas = dnt.calculate_density.calculate_all_surface_areas(spots_dfs, stems, all_mmfs, ap_vals)
cycle_relative_densities = dnt.calculate_density.calculate_relative_densities(spots_dfs, stems, all_mmfs, all_surface_areas, condition_map, cycles)

### 2D: fallout and late arrival

In [ ]:
df = spots_dfs[0].copy()

df = df.query("AP < 0.98 and status != 6")

df["fallout"] = df["distance_from_surface"] < -8

t = df.groupby("tracklet_id").agg({
    "cycle": "first",
    "fallout": "last",
    "id_kept": "last",
    "parent_id": "first",
    "track_id": "first",
    "frame": "count",
})

t["parent_tracklet"] = t["parent_id"].map(df["tracklet_id"])
t["num_children"] = t.index.map(t["parent_tracklet"].value_counts()).fillna(0).astype(int)

fallout_tracklets = t[t["fallout"]].index
no_children_tracklets = t[t["num_children"] == 0].index
started_fallout_tracklets = t[df.groupby("tracklet_id")["fallout"].first() == 1].index
no_parent_late_tracklets = t[(t["parent_id"] == -1) & (t["cycle"] >= 12)].index
parent_one_child_tracklets = t[t["parent_tracklet"].map(t["num_children"]) == 1].index

df_fallout = df.query("tracklet_id.isin(@fallout_tracklets) and "
                      "tracklet_id.isin(@no_children_tracklets) and "
                      "AP < 0.97 and "
                      "not tracklet_id.isin(@started_fallout_tracklets) and "
                      "not tracklet_id.isin(@no_parent_late_tracklets) and "
                      "not tracklet_id.isin(@parent_one_child_tracklets)").copy()

t_fallout_frame = df_fallout.groupby("tracklet_id")["fallout"].idxmax().map(df_fallout["time_since_nc11"])
t_fallout_ap = df_fallout.groupby("tracklet_id")["fallout"].idxmax().map(df_fallout["AP"])
t_fallout_cycle = df_fallout.groupby("tracklet_id")["fallout"].idxmax().map(df_fallout["cycle"])

t_arrival = df.groupby("track_id")["time_since_nc11"].min()
valid = (t_arrival < 5) * (df.groupby("track_id")["frame"].min() > df["frame"].min())
t_arrival = t_arrival[valid]
t_AP = df.groupby("track_id")["AP"].first()[valid]

fig, ax = plt.subplots(1, 1, figsize=(3, 2.25))
sns.scatterplot(y=t_fallout_frame, x=t_fallout_ap, color="#0a9396", edgecolor="k", )
sns.scatterplot(y=t_arrival, x=t_AP, color="#ee9b00", edgecolor="k", )
ax.spines[["top", "right"]].set_visible(False)

plt.xlabel("AP position")
plt.ylabel("Time since NC11 (min)")
plt.savefig(save_path / "fallout_and_arrival_by_ap.png", dpi=300, bbox_inches="tight")
plt.show()

### 2E fallout by cycle

In [ ]:
df = spots_dfs[0].copy()

df = df.query("AP < 0.98 and status != 6")

df["fallout"] = df["distance_from_surface"] < -8

t = df.groupby("tracklet_id").agg({
    "cycle": "first",
    "fallout": "last",
    "id_kept": "last",
    "parent_id": "first",
    "track_id": "first",
    "frame": "count",
})

t["parent_tracklet"] = t["parent_id"].map(df["tracklet_id"])
t["num_children"] = t.index.map(t["parent_tracklet"].value_counts()).fillna(0).astype(int)

fallout_tracklets = t[t["fallout"]].index
no_children_tracklets = t[t["num_children"] == 0].index
started_fallout_tracklets = t[df.groupby("tracklet_id")["fallout"].first() == 1].index
no_parent_late_tracklets = t[(t["parent_id"] == -1) & (t["cycle"] >= 12)].index
parent_one_child_tracklets = t[t["parent_tracklet"].map(t["num_children"]) == 1].index

df_filtered = df.query("AP < 0.97 and "
                      "not tracklet_id.isin(@started_fallout_tracklets) and "
                      "not tracklet_id.isin(@no_parent_late_tracklets) and "
                      "not tracklet_id.isin(@parent_one_child_tracklets)").copy()

t_fallout = df_filtered.groupby("tracklet_id")["fallout"].any()
t_cycle = df_filtered.groupby("tracklet_id")["cycle"].last()
t_first = df_filtered.groupby("tracklet_id")["parent_id"].first() == -1

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
sns.barplot(x=t_cycle, y=t_fallout, errorbar=None, edgecolor="k", color="#0a9396")
plt.xlabel("Cycle")
plt.ylabel("Rate of fallout")
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / "fallout_by_cycle.png", dpi=300, bbox_inches="tight")
plt.show()



### 2F Lineage count at NC 14

In [ ]:
df = spots_dfs[0].copy()

df = df.query("AP < 0.98 and status != 6")

df["fallout"] = df["distance_from_surface"] < -8

t = df.groupby("tracklet_id").agg({
    "cycle": "first",
    "fallout": "last",
    "id_kept": "last",
    "parent_id": "first",
    "track_id": "first",
    "frame": "count",
})

t["parent_tracklet"] = t["parent_id"].map(df["tracklet_id"])
t["num_children"] = t.index.map(t["parent_tracklet"].value_counts()).fillna(0).astype(int)

no_parent_late_tracklets = t[(t["parent_id"] == -1) & (t["cycle"] >= 12)].index
parent_one_child_tracklets = t[t["parent_tracklet"].map(t["num_children"]) == 1].index

df_filtered = df.query("AP < 0.97 and "
                      "not tracklet_id.isin(@no_parent_late_tracklets) and "
                      "not tracklet_id.isin(@parent_one_child_tracklets)").copy()

track_id_first_cycle = df_filtered.groupby("track_id")["cycle"].first()
track_id_first_frame = df_filtered.groupby("track_id")["frame"].first()
track_id_before_12 = track_id_first_cycle[track_id_first_cycle < 12].index

df2 = df_filtered.query("frame == frame.max() and track_id.isin(@track_id_before_12)").copy()
df2["first_cycle"] = df2["track_id"].map(track_id_first_cycle)
df2["first_frame"] = df2["track_id"].map(track_id_first_frame)
t = df2.groupby("track_id").agg({
    "frame": "count",
    "first_cycle": "first",
    "first_frame": "first",
})

t["cycle"] = (t["first_frame"] > 26) + 10
t = t[t["cycle"] == 10]

fig, ax = plt.subplots(1, 1, figsize=(1.8, 1.8))
sns.histplot(t, x="frame", hue="cycle", binwidth=1.0, palette={10:"#ee9b00", 11: "#ae2012"}, multiple="stack", hue_order=[11, 10], edgecolor="k", binrange=(0.5, 16.5), stat="probability", alpha=1.0, legend=False)
ax = plt.gca()
ax.spines[["top", "right"]].set_visible(False)
plt.xlabel("# of nuclei in NC14")
plt.xticks([0, 4, 8, 12, 16])
plt.ylabel("Fraction of lineages")
plt.savefig(save_path / "lineage_count_nc14.png", dpi=300, bbox_inches="tight")
plt.show()

### 2h: transfer matrix

In [ ]:
all_transfer_matrices = []

k = 0


df = spots_dfs[k].query("AP_bin > 0.001 and AP_bin <= 1.0").copy()

initial_frame = all_mmfs[stems[k]][0]
final_frame = all_mmfs[stems[k]][-1]
surface_area = all_surface_areas[stems[k]]


initial_vec = df.query("frame == @initial_frame").groupby("AP_bin")["AP"].count().to_numpy()[:-1]


final_frame_df = df.query("frame == @final_frame").copy()
initial_frame_df = df.query("frame == @initial_frame").copy()

final_frame_df["initial_AP_bin"] = final_frame_df["track_id"].map(initial_frame_df.set_index("track_id")["AP_bin"])


transfer_matrix = final_frame_df.groupby("AP_bin")["initial_AP_bin"].value_counts().unstack().fillna(0)


transfer_matrix = transfer_matrix.iloc[:, :].to_numpy()

transfer_matrix_per_nuc = transfer_matrix / initial_vec

annot_labels = []
for row in range(transfer_matrix_per_nuc.shape[0]):
    annot_labels.append([])

    for col in range(transfer_matrix_per_nuc.shape[1]):
        value = transfer_matrix_per_nuc[row, col]
        if value > 0.05:
            annot_labels[-1].append(f"{value:.1f}")
        else:
            annot_labels[-1].append("")

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
cbar_fig, cbar_ax = plt.subplots(1, 1, figsize=(0.5, 3))
sns.heatmap(
    transfer_matrix_per_nuc,
    xticklabels=[],
    yticklabels=[],
    linewidths=0.5,
    linecolor="k",
    fmt="",
    annot=annot_labels,
    cmap="Reds",
    cbar_kws={"label": ""},
    ax=ax,
    cbar_ax=cbar_ax,
    square=True)

fig.savefig(save_path / f"{stems[k][:-6]}_ap_bin_transfer_matrix.png", dpi=300, bbox_inches="tight")
cbar_fig.savefig(save_path / f"{stems[k][:-6]}_ap_bin_transfer_matrix_colorbar.png", dpi=300, bbox_inches="tight")
plt.show()


### 2G lineage displacement

In [ ]:
k = 0
colors = ["#0a9396", "#ee9b00", "#ae2012"]


fig, ax = plt.subplots(1, 1, figsize=(2.5, 2.2))
df = spots_dfs[k].copy()
first_frame = all_mmfs[stems[0]][0]
df = df[df["frame"] >= first_frame].copy()
df["initial_position"] = df.groupby("track_id")["AP"].transform("first")
df["displacement_from_start"] = (df["AP"] - df["initial_position"]) * 100
df["initial_cycle"] = df.groupby("track_id")["cycle"].transform("first")

df = df[df["frame"] == all_mmfs[stems[0]][-1]].copy()
df = df.query("initial_cycle == 10")
t = df.groupby("track_id")[["AP", "displacement_from_start", "initial_cycle"]].mean().reset_index()

track_id_colors = {}
frame_df = df.query("frame == frame.min()")
frame_positions = frame_df.groupby("track_id")[["AP"]].mean()
for position, color in zip([0.15, 0.5, 0.9], colors):
    distances = (frame_positions["AP"] - position)**2
    print(t.set_index("track_id").loc[distances.idxmin(), "AP"])
    track_id_colors[distances.idxmin()] = color

is_special = np.array([tid in track_id_colors.keys() for tid in t.track_id])

sns.scatterplot(t[~is_special], x="AP", y="displacement_from_start", color="#D8D4D1", edgecolor="k", legend=False, s=25, lw=1.0, alpha=0.7)
sns.scatterplot(t[is_special], x="AP", y="displacement_from_start", hue="track_id", palette=track_id_colors, edgecolor="k", legend=False, s=45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xlabel("AP Position")
plt.ylabel("Displacement along AP axis")
plt.axhline(0, linestyle="--", linewidth=2, color="k")
plt.savefig(save_path / f"{stems[k]}_dislacement.png", dpi=300, bbox_inches="tight")
plt.show()